Organize by Temporal Phase + Merge (<30 min apart)

In [1]:
import os
import re
import glob
import shutil
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds

In [2]:
# ================== User Settings ==================
INPUT_FOLDER = r"/Users/ks/Desktop/Wu/testing1"   # folder full of ECOSTRESS tiffs

MERGE_WINDOW_MIN = 5   # merge files less than this many minutes apart

# Matches: ECO_L2T_LSTE.002_cloud_20180802T230140_aid0001_11N.tif
#          ECO_L2T_LSTE.002_LST_err_20180802T230140_aid0001_11N.tif
FILENAME_PATTERN = re.compile(
    r'^ECO_L2T_LSTE\.(?P<version>\d{3})_'
    r'(?P<band>LST_err|LST|QC|cloud|water|height|EmisWB)_'
    r'(?P<timestamp>\d{8}T\d{6})_'
    r'aid(?P<aid>\d+)_'
    r'(?P<utm_zone>\d{1,2}[A-Z])'
    r'\.tiff?$',
    re.IGNORECASE
)

PHASE_CATEGORIES = ["Night", "Morning", "Afternoon", "Evening"]

# Bands that should always travel together as one "acquisition group"
BAND_NAMES = ["LST", "LST_err", "QC", "cloud", "water", "height", "EmisWB"]

In [3]:
def parse_ecostress_filename(path):
    """
    Parses an ECO_L2T_LSTE filename into its components.
    Returns a dict: {version, band, dt_utc, aid, utm_zone} or None if malformed.
    """
    filename = os.path.basename(path)
    match = FILENAME_PATTERN.match(filename)
    if not match:
        print(f"Skipping malformed filename: {filename}")
        return None

    try:
        dt_utc = datetime.strptime(match.group("timestamp"), "%Y%m%dT%H%M%S").replace(
            tzinfo=ZoneInfo("UTC")
        )
    except ValueError:
        print(f"Could not parse timestamp in: {filename}")
        return None

    return {
        "version": match.group("version"),
        "band": match.group("band"),
        "dt_utc": dt_utc,
        "aid": match.group("aid"),
        "utm_zone": match.group("utm_zone"),
        "path": path,
    }


def parse_timestamp_from_name(path):
    """Kept for backwards compatibility with downstream code that just wants the datetime."""
    parsed = parse_ecostress_filename(path)
    return parsed["dt_utc"] if parsed else None


def classify_time(hour):
    """Classify an hour (0-23, local time) into a temporal phase."""
    if 0 <= hour < 5:
        return "Night"
    elif 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    else:
        return "Evening"


def group_files_by_acquisition(folder_path):
    """
    Groups all band files in folder_path by acquisition (same timestamp + aid + utm_zone),
    since each scene now produces separate LST/QC/cloud/water/height/EmisWB files that
    need to stay linked for later QA masking.

    Returns: {(timestamp_str, aid, utm_zone): {band: filepath, ...}}
    """
    groups = {}
    for filename in os.listdir(folder_path):
        if not filename.lower().endswith((".tif", ".tiff")):
            continue

        fpath = os.path.join(folder_path, filename)
        parsed = parse_ecostress_filename(fpath)
        if parsed is None:
            continue

        key = (parsed["dt_utc"].strftime("%Y%m%dT%H%M%S"), parsed["aid"], parsed["utm_zone"])
        groups.setdefault(key, {})[parsed["band"]] = fpath

    return groups

In [4]:
def organize_tifs_by_phase(folder_path, local_tz="America/Los_Angeles"):
    """
    Groups files by acquisition (all bands together), then moves each whole
    acquisition group into a Night/Morning/Afternoon/Evening subfolder based
    on its local (Pacific) capture time. Keeps LST/QC/cloud/water/etc. files
    for the same scene together in the same destination folder.

    Returns dict of {phase: [(timestamp_str, band_dict), ...]}
    """
    for cat in PHASE_CATEGORIES:
        os.makedirs(os.path.join(folder_path, cat), exist_ok=True)

    classified = {cat: [] for cat in PHASE_CATEGORIES}

    acquisitions = group_files_by_acquisition(folder_path)

    for (ts_str, aid, utm_zone), band_files in acquisitions.items():
        dt_utc = datetime.strptime(ts_str, "%Y%m%dT%H%M%S").replace(tzinfo=ZoneInfo("UTC"))
        dt_local = dt_utc.astimezone(ZoneInfo(local_tz))
        phase = classify_time(dt_local.hour)

        moved_band_files = {}
        for band, source_path in band_files.items():
            filename = os.path.basename(source_path)
            dest_path = os.path.join(folder_path, phase, filename)
            shutil.move(source_path, dest_path)
            moved_band_files[band] = dest_path

        classified[phase].append(
            (dt_local.strftime("%Y-%m-%d %H:%M:%S %Z"), moved_band_files)
        )

    return classified

In [5]:
def group_by_time(files, max_minutes=30):
    """
    Chains files sorted by timestamp into groups where each file is within
    max_minutes of the PREVIOUS file in the group (so a chain of near files
    can span more than max_minutes total, but every consecutive gap is small).
    """
    max_delta = timedelta(minutes=max_minutes)

    items = [(parse_timestamp_from_name(f), f) for f in files]
    items = [(dt, f) for dt, f in items if dt is not None]
    items.sort(key=lambda x: x[0])

    groups = []
    current_group = []

    for dt, f in items:
        if not current_group:
            current_group = [(dt, f)]
            continue

        prev_dt = current_group[-1][0]
        if dt - prev_dt <= max_delta:
            current_group.append((dt, f))
        else:
            groups.append(current_group)
            current_group = [(dt, f)]

    if current_group:
        groups.append(current_group)

    return [[f for _, f in grp] for grp in groups]


def mosaic_mean(files, out_path):
    """Mosaic a list of GeoTIFFs using the mean for overlapping pixels."""
    if len(files) == 0:
        return

    datasets = [rasterio.open(f) for f in files]
    ref = datasets[0]
    ref_crs = ref.crs

    xres, yres = ref.res
    xres, yres = abs(xres), abs(yres)

    minxs, maxxs, minys, maxys = [], [], [], []
    for ds in datasets:
        if ds.crs != ref_crs:
            raise ValueError("All rasters must have the same CRS.")
        b = ds.bounds
        minxs.append(min(b.left, b.right))
        maxxs.append(max(b.left, b.right))
        minys.append(min(b.bottom, b.top))
        maxys.append(max(b.bottom, b.top))

    minx, maxx = min(minxs), max(maxxs)
    miny, maxy = min(minys), max(maxys)

    width = int(np.ceil((maxx - minx) / xres))
    height = int(np.ceil((maxy - miny) / yres))
    mosaic_transform = from_bounds(minx, miny, maxx, maxy, width, height)

    n = len(datasets)
    stack = np.full((n, height, width), np.nan, dtype="float32")

    for i, ds in enumerate(datasets):
        data = ds.read(1).astype("float32")
        if ds.nodata is not None and not np.isnan(ds.nodata):
            data = np.where(data == ds.nodata, np.nan, data)

        dest = np.full((height, width), np.nan, dtype="float32")
        reproject(
            source=data,
            destination=dest,
            src_transform=ds.transform,
            src_crs=ds.crs,
            dst_transform=mosaic_transform,
            dst_crs=ref_crs,
            src_nodata=None,
            dst_nodata=np.nan,
            resampling=Resampling.nearest,
        )
        stack[i] = dest

    mosaic_arr = np.nanmean(stack, axis=0).astype("float32")

    meta = ref.meta.copy()
    meta.update({
        "height": height,
        "width": width,
        "transform": mosaic_transform,
        "dtype": "float32",
        "nodata": np.nan,
    })

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(mosaic_arr, 1)

    for ds in datasets:
        ds.close()

In [8]:
def process_folder(input_folder, merge_window_min=MERGE_WINDOW_MIN):
    print(f"Organizing tiffs in {input_folder} by temporal phase...")
    classified = organize_tifs_by_phase(input_folder)

    for phase, entries in classified.items():
        print(f"\n{phase}: {len(entries)} acquisition(s)")
        for ts, band_files in entries:
            bands_str = ", ".join(sorted(band_files.keys()))
            print(f"  {ts}  -> [{bands_str}]")

    for phase in PHASE_CATEGORIES:
        phase_folder = os.path.join(input_folder, phase)

        # re-group the now-moved files in this phase folder by acquisition
        acquisitions = group_files_by_acquisition(phase_folder)
        if not acquisitions:
            continue

        # build one (dt_utc, band_dict) entry per acquisition, sorted by time
        acq_list = []
        for (ts_str, aid, utm_zone), band_files in acquisitions.items():
            dt_utc = datetime.strptime(ts_str, "%Y%m%dT%H%M%S").replace(tzinfo=ZoneInfo("UTC"))
            acq_list.append((dt_utc, band_files))
        acq_list.sort(key=lambda x: x[0])

        # group acquisitions in time (same logic as group_by_time, but on acquisitions not files)
        max_delta = timedelta(minutes=merge_window_min)
        groups = []
        current_group = []
        for dt_utc, band_files in acq_list:
            if not current_group:
                current_group = [(dt_utc, band_files)]
                continue
            prev_dt = current_group[-1][0]
            if dt_utc - prev_dt <= max_delta:
                current_group.append((dt_utc, band_files))
            else:
                groups.append(current_group)
                current_group = [(dt_utc, band_files)]
        if current_group:
            groups.append(current_group)

        print(f"\n[{phase}] {len(groups)} merge group(s) from {len(acq_list)} acquisition(s)")

        merged_folder = os.path.join(phase_folder, "merged")

        for i, group in enumerate(groups, start=1):
            if len(group) == 1:
                # nothing to merge, leave as-is
                continue

            dts = [dt for dt, _ in group]
            first_dt, last_dt = min(dts), max(dts)
            ts1 = first_dt.strftime("%Y%m%dT%H%M%S")
            ts2 = last_dt.strftime("%Y%m%dT%H%M%S")

            # merge each band separately (LST files mosaic with LST files, QC with QC, etc.)
            for band in BAND_NAMES:
                band_files = [band_dict[band] for _, band_dict in group if band in band_dict]
                if len(band_files) < 2:
                    continue  # nothing to merge for this band in this group

                out_name = f"{phase}_group{i:03d}_{ts1}_to_{ts2}_{band}_mean.tif"
                out_path = os.path.join(merged_folder, out_name)

                print(f"  Merging {len(band_files)} {band} file(s) -> {out_name}")
                for f in band_files:
                    print(f"    {os.path.basename(f)}")

                mosaic_mean(band_files, out_path)

    print("\nDone.")

In [9]:
process_folder(INPUT_FOLDER, merge_window_min=MERGE_WINDOW_MIN)

Organizing tiffs in /Users/ks/Desktop/Wu/testing1 by temporal phase...

Night: 0 acquisition(s)

Morning: 0 acquisition(s)

Afternoon: 0 acquisition(s)

Evening: 0 acquisition(s)

[Afternoon] 4 merge group(s) from 8 acquisition(s)
  Merging 2 cloud file(s) -> Afternoon_group001_20180802T230140_to_20180802T230232_cloud_mean.tif
    ECO_L2T_LSTE.002_cloud_20180802T230140_aid0001_11N.tif
    ECO_L2T_LSTE.002_cloud_20180802T230232_aid0001_11N.tif


/var/folders/kn/0k02zq75745fz70kszyd1wjh0000gn/T/ipykernel_82941/3515311814.py:85: RuntimeWarning: Mean of empty slice
  mosaic_arr = np.nanmean(stack, axis=0).astype("float32")
/var/folders/kn/0k02zq75745fz70kszyd1wjh0000gn/T/ipykernel_82941/3515311814.py:85: RuntimeWarning: Mean of empty slice
  mosaic_arr = np.nanmean(stack, axis=0).astype("float32")


  Merging 2 cloud file(s) -> Afternoon_group002_20180805T220222_to_20180805T220314_cloud_mean.tif
    ECO_L2T_LSTE.002_cloud_20180805T220222_aid0001_11N.tif
    ECO_L2T_LSTE.002_cloud_20180805T220314_aid0001_11N.tif
  Merging 2 cloud file(s) -> Afternoon_group003_20180806T211046_to_20180806T211138_cloud_mean.tif
    ECO_L2T_LSTE.002_cloud_20180806T211046_aid0001_11N.tif
    ECO_L2T_LSTE.002_cloud_20180806T211138_aid0001_11N.tif
  Merging 2 cloud file(s) -> Afternoon_group004_20180809T201128_to_20180809T201220_cloud_mean.tif
    ECO_L2T_LSTE.002_cloud_20180809T201128_aid0001_11N.tif
    ECO_L2T_LSTE.002_cloud_20180809T201220_aid0001_11N.tif

[Evening] 2 merge group(s) from 2 acquisition(s)

Done.


/var/folders/kn/0k02zq75745fz70kszyd1wjh0000gn/T/ipykernel_82941/3515311814.py:85: RuntimeWarning: Mean of empty slice
  mosaic_arr = np.nanmean(stack, axis=0).astype("float32")
/var/folders/kn/0k02zq75745fz70kszyd1wjh0000gn/T/ipykernel_82941/3515311814.py:85: RuntimeWarning: Mean of empty slice
  mosaic_arr = np.nanmean(stack, axis=0).astype("float32")
